# 10 — Tamamlanan final testinin salt okunur denetimi

**Sürüm 2.** Notebook yalnızca komutları düzenler; araştırma kodu Python modüllerindedir.
Colab'ın Python/Torch paketleri araştırma için kullanılmaz. Sabit Python 3.13.15 ve
hashli paket kilidiyle ayrı süreç ortamı kurulur. İlk kurulum ve GPU paketleri zaman alabilir.

**Eski `colab_v1` arşivi salt okunurdur.** Bu notebook eski sonuçları güncellemez.
06/07 varsayılanı eksik iki tabloyu yeniden oluşturup özgün SHA ile karşılaştırmaktır;
başarılı dosyalar ayrı recovery klasörüne yazılır. Bu bir yeniden eğitim değildir.
Yeni eğitim için 06/07'de MODE='research', ortak WORKSPACE_ID ve ALLOW_TRAINING=True gerekir.
08/09 eski kontrol sonuçlarını yeni ortamla birleştirmez; aynı yeni workspace'in
A–D sonuçlarını kullanır. 10, görülmüş Eylül 2026 testini yeniden çalıştırmaz.

**Akış:** ayarlar → Drive → sabit Git sürümü → izole ortam → veri/işlem → durum → yedek.
Aynı workspace'i iki Colab oturumunda eşzamanlı çalıştırmayın.


## 1 — Çalışma ayarları

Aynı deneye devam ederken WORKSPACE_ID ve sürümü değiştirmeyin.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, subprocess, sys

MODE = 'archive'  # recovery: yalnızca tablolar; research: yeni deney; archive: finali oku
PHASE = 'final'
WORKSPACE_ID = 'advisor_protocol_v2'
EXPERIMENTS = []
ALLOW_TRAINING = False
RELEASE_TAG = 'advisor-notebooks-v2.0.0'  # Sabit yayın; dalın güncel ucunu otomatik çekmez.
REPO_URL = 'https://github.com/umutergul74/yeniBot.git'
REPO_DIR = Path('/content/advisor_reproducible_repo')
RUNTIME_DIR = Path('/content/advisor_isolated_runtime')
DRIVE_BASE = Path('/content/drive/MyDrive/yeniBot/advisor_experiments')
SESSION = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
assert WORKSPACE_ID.replace('_','').replace('-','').isalnum()
assert MODE in ('recovery','research','archive')
assert PHASE != 'final' or MODE == 'archive', 'Görülmüş final testi yeniden çalıştırılmaz.'


## 2 — Drive ve ayrı çıktı alanı

Hazırlanmış tablolar, ağırlıklar ve ilerleme kalıcı alana yazılır.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
LEGACY_ROOT = DRIVE_BASE / 'colab_v1'
namespace = {'recovery':'recovery_v2','research':'research_v2','archive':'archive_audit_v2'}[MODE]
ROOT = DRIVE_BASE / namespace / WORKSPACE_ID
ROOT.mkdir(parents=True,exist_ok=True)
LOG_DIR = ROOT / 'sessions' / SESSION
LOG_DIR.mkdir(parents=True,exist_ok=True)
print('Eski arşiv (salt okunur):',LEGACY_ROOT)
print('Bu çalışmanın kalıcı alanı:',ROOT)
print('Oturum günlükleri:',LOG_DIR)


## 3 — Sabit Git yayını

Yerel değişiklik varsa otomatik silinmez; yanlış sürümle devam edilmez.


In [ ]:
if not (REPO_DIR/'.git').exists():
    subprocess.run(['git','clone','--branch',RELEASE_TAG,'--single-branch',REPO_URL,str(REPO_DIR)],check=True)
def git(*args):
    return subprocess.check_output(['git','-C',str(REPO_DIR),*args],text=True).strip()
assert not git('status','--porcelain','--untracked-files=no'), 'Kodda yerel değişiklik var.'
assert git('rev-parse','HEAD') == git('rev-parse',RELEASE_TAG+'^{commit}'), 'Yanlış yayın; yeni runtime açın.'
print('Kod sürümü:',git('rev-parse','HEAD'))


## 4 — Kilitli araştırma ortamı

Sistem Python'una pip install yapılmaz. Ayrı Python + venv kullanılır; bütün
bağımlılıklar sürüm ve SHA ile sabittir. Drive'daki indirme önbelleği tekrar kullanılır.
Kurulum başarısızsa sonraki hücreye geçmeyin. Eğitim için Tesla T4 seçin.


In [ ]:
def logged(command, name):
    with (LOG_DIR/name).open('a',encoding='utf-8',buffering=1) as log:
        env=dict(os.environ)
        for key in ('PYTHONPATH','PYTHONHOME','PYTHONSTARTUP','IPYTHONDIR','JUPYTER_CONFIG_DIR'):
            env.pop(key,None)
        env['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
        with subprocess.Popen(command,cwd=REPO_DIR,env=env,stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line,end=''); log.write(line)
                code=process.wait()
            except BaseException:
                process.terminate(); process.wait(); raise
            if code: raise subprocess.CalledProcessError(code,command)
RECEIPT=RUNTIME_DIR/'environment_receipt.json'
logged([sys.executable,'-I',str(REPO_DIR/'scripts/bootstrap_advisor.py'),
        '--repo',str(REPO_DIR),'--runtime',str(RUNTIME_DIR),
        '--cache',str(DRIVE_BASE/'reproducibility_cache_v2'),
        '--receipt',str(RECEIPT)],'environment_setup.log')
runtime=json.loads(RECEIPT.read_text())
RESEARCH_PYTHON=runtime['python']
print('İzole Python:',RESEARCH_PYTHON)
print('Paket kilidi:',runtime['lock_sha256'])

def execute_step(action):
    command=[RESEARCH_PYTHON,'-I',str(REPO_DIR/'scripts/advisor_workspace.py'),action,
             '--root',str(ROOT),'--legacy',str(LEGACY_ROOT),'--mode',MODE,
             '--phase',PHASE,'--session',SESSION,'--experiments',*EXPERIMENTS]
    if ALLOW_TRAINING: command.append('--allow-training')
    logged(command,action+'.log')


## 5 — Arşivlenen finali doğrula

Model/scaler, fit/test tablo ve tahmin hashleri kontrol edilir. Ağırlıklar
yüklenmez; predict/evaluate/fit çağrılmaz. Sonuç ayrı rapor alanına yazılır.
Yeni final döneminin kullanımı ayrı bilimsel protokol gerektirir.


In [ ]:
execute_step('audit-final')


## 7 — İlerleme ve çıktıların yeri


In [ ]:
execute_step('status')
print('Durum:',ROOT/'STATUS.md')
print('Hazırlanmış tablolar:',ROOT/'data')
print('Checkpointler:',ROOT/'runs')


## 8 — İnceleme paketi ve tam dosya envanteri

Küçük inceleme ZIP'i modelleri içermez. Prepared ZIP'i tabloları içerir.
Ağırlıklar/raw veriler Drive klasöründedir; tam yedek için workspace
klasörünün tamamını indirin. `full_artifact_inventory.json` tüm dosyaların
hashlerini listeler; aktif oturum loglarini ayri belirtir. Runtime kapanırsa notebooku baştan çalıştırın.
Sert VM kapanmasından sonra `.writer_lease` kalmışsa başka yazıcı
olmadığını doğrulamadan kilidi kaldırmayın. Ortam/sürüm değişirse eski
çalışma sessizce devam etmez. Platformlar arasında bit eşitliği garanti değildir.


In [ ]:
execute_step('bundle')
print('İnceleme paketi:',ROOT/'reports/advisor_latest_review_bundle.zip')
print('Tam envanter:',ROOT/'reports/full_artifact_inventory.json')
